# Lasana Job 5 — vanilla SegFormer-B0, seed 44

**Person 4 (Lasana)** assigned run from `Phase2/Dhinanjaya-Person5/multiseed_kickoff.md`.
This notebook is pre-filled for Job 5 — do **not** change the Config cell.

- Variant: vanilla SegFormer-B0 (no attention loss)
- Seed: 44 (extra seed for Table 1 mean±std)
- Drive output: `MyDrive/multiseed_outputs_vanilla_seed44/`

Upload **only** `multiseed_train_colab.zip` (build locally via
`python Phase2/Dhinanjaya-Person5/make_multiseed_colab_zip.py`), not the whole repo.

**Before running:** Runtime → Change runtime type → GPU (T4 or better).
Upload the zip to `MyDrive/multiseed_train_colab.zip`, then **Run all** (~2h).

When finished, share the whole `MyDrive/multiseed_outputs_vanilla_seed44/` folder
(or zip it) back to Dhinanjaya — do **not** merge results into the repo yourself.

## Config — pre-filled for Lasana Job 5 (do not edit)

In [ ]:
# --- Lasana Job 5 (pre-filled — do not change) ---
VARIANT = "vanilla"        # "vanilla" or "att"
SEED = 44                  # 43 or 44 (or 42 for the KL job)
LAMBDA2 = 1.0              # ignored for vanilla; kept for shared notebook parity
ATT_MODE = "mse"           # ignored for vanilla
LAMBDA3 = 0.0              # 0.0 for non-boundary jobs
OUTPUT_TAG = "vanilla_seed44"  # Drive folder: multiseed_outputs_vanilla_seed44
# ------------------------------------------

## Step 0: Unzip + deps

In [ ]:
import sys, zipfile, shutil
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
ZIP_ON_DRIVE = Path("/content/drive/MyDrive/multiseed_train_colab.zip")
BUNDLE = Path("/content/multiseed_train")
OUTPUTS = Path(f"/content/drive/MyDrive/multiseed_outputs_{OUTPUT_TAG}")

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    if not (BUNDLE / "paths.py").exists():
        if not ZIP_ON_DRIVE.is_file():
            raise FileNotFoundError(f"Upload zip to {ZIP_ON_DRIVE}")
        print("Unzipping", ZIP_ON_DRIVE)
        with zipfile.ZipFile(ZIP_ON_DRIVE, "r") as z:
            z.extractall(BUNDLE.parent)
        if not (BUNDLE / "paths.py").exists():
            cands = [p for p in BUNDLE.parent.iterdir() if p.is_dir() and (p / "paths.py").exists()]
            if not cands:
                raise FileNotFoundError("paths.py not found after unzip")
            if BUNDLE.exists():
                shutil.rmtree(BUNDLE)
            shutil.move(str(cands[0]), str(BUNDLE))
    HERE = BUNDLE
else:
    HERE = Path.cwd()
    OUTPUTS = HERE / f"outputs_{OUTPUT_TAG}"

sys.path.insert(0, str(HERE))
OUTPUTS.mkdir(parents=True, exist_ok=True)
print("HERE =", HERE)
print("OUTPUTS =", OUTPUTS)
print(f"Job: variant={VARIANT} seed={SEED} lambda2={LAMBDA2} att_mode={ATT_MODE} lambda3={LAMBDA3}")

In [ ]:
import subprocess, sys
pkgs = ["transformers", "accelerate", "thop", "tqdm", "opencv-python-headless"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])
print("deps ready")

## Step 1: Device check

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
if not torch.cuda.is_available():
    print("WARNING: no GPU -- this job is ~2h on T4; CPU will not finish in reasonable time.")

## Step 2: Train (this job only — ~2h on T4)

In [ ]:
import paths
from paths import add_teammate_paths, apply_data_dirs

paths.set_output_dirs(OUTPUTS / "checkpoints", OUTPUTS / "results")
add_teammate_paths()
apply_data_dirs()
print("CKPT_DIR:", paths.CKPT_DIR)
print("RESULTS_DIR:", paths.RESULTS_DIR)
assert paths.DATA_IMG_DIR.is_dir() and paths.DATA_MASK_DIR.is_dir()

import train_full_scale as T
import torch

class Args:
    n_train, n_val, n_test = 3576, 766, 766
    epochs = 20
    batch_size = 16
    lr = 6e-5
    lambda2 = LAMBDA2
    sigma = 8.0
    att_mode = ATT_MODE
    lambda3 = LAMBDA3
    boundary_kernel = 3
    seed = SEED

args = Args()
torch.manual_seed(args.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(args.seed)

best = paths.CKPT_DIR / f"segformer_b0_{VARIANT}_best.pt"
if best.exists():
    print(f"SKIP: {best} already exists")
else:
    T.train_variant(VARIANT, args)

## Step 3: Evaluate + collect results

In [ ]:
import eval_full_scale as E

class EvalArgs:
    n_train, n_val, n_test = 3576, 766, 766
    seed = SEED

row = E.evaluate_variant(VARIANT, EvalArgs())
print(f"{row['model']}: dice={row['dice']} iou={row['iou']} aamo={row['aamo']}")

print("\nOutputs under", OUTPUTS)
for p in sorted(OUTPUTS.rglob("*")):
    if p.is_file():
        print(" ", p.relative_to(OUTPUTS))

## Step 4: Send back (Lasana Job 5)

Share the whole `MyDrive/multiseed_outputs_vanilla_seed44/` folder (or zip it)
back to Dhinanjaya. Required files:

- `checkpoints/segformer_b0_vanilla_best.pt`
- `results/train_summary_vanilla.json`
- `results/training_log_vanilla.csv`
- `results/eval_vanilla.json`
- `results/baseline_comparison.csv`

Do **not** merge these into the repo yourself — Dhinanjaya aggregates all 7 jobs.